# Evaluation Metrics

This notebook accompanies the note [Evaluation Metrics](../../notes/resampling_and_model_assessment/metrics.md). We compute classification metrics from a confusion matrix, then regression metrics (MAE, MSE, RMSE, $R^2$) on a noisy sine wave fitted with a straight line, and finally see how outliers and noise level change them.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score)

rng = np.random.default_rng(42)

## Classification metrics from a confusion matrix

With $TP=70$, $FN=30$, $FP=20$, $TN=80$ (the example in the note) the metrics follow directly from the four counts. Note that $F_1 = 2TP/(2TP+FP+FN)$.

In [2]:
TP, FN, FP, TN = 70, 30, 20, 80
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall = TP / (TP + FN)
specificity = TN / (TN + FP)
f1 = 2 * TP / (2 * TP + FP + FN)
print(f"accuracy={accuracy:.3f} precision={precision:.3f} recall={recall:.3f} "
      f"specificity={specificity:.3f} F1={f1:.3f}")

accuracy=0.750 precision=0.778 recall=0.700 specificity=0.800 F1=0.737


## Accuracy under class imbalance

A classifier that always predicts the majority class looks excellent on accuracy but has zero recall, which is why accuracy alone can mislead for rare positives.

In [3]:
y_true = np.r_[np.ones(10), np.zeros(990)]
y_pred = np.zeros(1000)
print("accuracy:", (y_true == y_pred).mean())
print("recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred, zero_division=0))
print(confusion_matrix(y_true, y_pred))

accuracy: 0.99
recall: 0.0
F1: 0.0
[[990   0]
 [ 10   0]]


## Regression metrics for a linear fit to a sine wave

We fit a straight line to $y=\sin x+\varepsilon$ on $[0,2\pi]$. The model is misspecified, so there is structured error that the metrics summarise differently.

In [4]:
n = 100
x = 2 * np.pi * rng.random(n)
y = np.sin(x) + rng.normal(0, 0.15, n)
model = LinearRegression().fit(x[:, None], y)
y_hat = model.predict(x[:, None])

mae = mean_absolute_error(y, y_hat)
mse = mean_squared_error(y, y_hat)
rmse = np.sqrt(mse)
r2 = r2_score(y, y_hat)
print(f"MAE={mae:.3f} MSE={mse:.3f} RMSE={rmse:.3f} R2={r2:.3f}")

grid = np.linspace(0, 2 * np.pi, 200)
plt.figure(figsize=(8, 5))
plt.scatter(x, y, s=15, label="data")
plt.plot(grid, model.predict(grid[:, None]), "r", label="linear fit")
plt.vlines(x, y, y_hat, colors="gray", linestyles="dotted", linewidth=0.7)
plt.title(f"Linear fit to a sine wave: RMSE={rmse:.2f}, R2={r2:.2f}")
plt.legend(); plt.show()

MAE=0.354 MSE=0.174 RMSE=0.417 R2=0.676


/tmp/ipykernel_84657/669499654.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


## Outlier sensitivity and negative $R^2$

Adding one extreme error inflates RMSE far more than MAE. Separately, $R^2$ is negative whenever predictions are worse than the constant $\bar y$.

In [5]:
e = np.array([0.5, -0.5, 0.0, -1.0])
e_out = e.copy(); e_out[0] += 10
for label, err in [("clean", e), ("one outlier", e_out)]:
    print(f"{label:12s} MAE={np.abs(err).mean():.3f} MSE={(err**2).mean():.3f} RMSE={np.sqrt((err**2).mean()):.3f}")

y_c = np.array([2, 4, 6, 8, 10.0])
print("R2 of constant 8:", r2_score(y_c, np.full(5, 8.0)))  # 1 - 60/40

clean        MAE=0.500 MSE=0.375 RMSE=0.612
one outlier  MAE=3.000 MSE=27.875 RMSE=5.280
R2 of constant 8: -0.5


## ROC curve and AUC

AUC equals the probability that a random positive receives a higher score than a random negative. We check the pair-counting definition against scikit-learn for the small example from the exercises.

In [6]:
pos = np.array([0.9, 0.8, 0.6, 0.4]); neg = np.array([0.7, 0.5, 0.3, 0.2])
pairs = (pos[:, None] > neg[None, :]).mean()
labels = np.r_[np.ones(4), np.zeros(4)]
scores = np.r_[pos, neg]
print("pair-counting AUC:", pairs, " sklearn AUC:", roc_auc_score(labels, scores))

thr = np.sort(np.unique(np.r_[scores, 1.0, 0.0]))[::-1]
tpr = [(pos >= t).mean() for t in thr]; fpr = [(neg >= t).mean() for t in thr]
plt.plot(fpr, tpr, "o-"); plt.plot([0, 1], [0, 1], "--", c="gray")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC curve"); plt.show()

pair-counting AUC: 0.8125  sklearn AUC: 0.8125


/tmp/ipykernel_84657/4080949767.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC curve"); plt.show()
